# Cross-topic Argument Mining with RoBERTa — Colab

Re-implementation of Stab et al. (EMNLP 2018), *Cross-topic Argument Mining from Heterogeneous Sources*,
with **RoBERTa replacing the Contextual BiLSTM (BiCLSTM)**. Topic information is fed as
the first segment of a sentence pair (`<s> topic </s></s> sentence </s>`).

Section N runs **four blocks in this order**:

1. **MTL + DIP2016 / 2-label**  (RoBERTa-MTL = shared encoder + UKP head + DIP relevance head)
2. **MTL + DIP2016 / 3-label**
3. **single-task / 2-label**  (plain `RobertaForSequenceClassification`)
4. **single-task / 3-label**

Each block sweeps all held-out topics and all seeds.

A green ✓ appears next to every dataset (held-out topic) as soon as its last seed
finishes training and testing. All metrics, predictions, confusion matrices,
classification reports, and Table-4 layout tables are saved to Drive as CSV +
PNG.

## A. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## B. Install dependencies

In [ ]:
!pip install -q "transformers>=4.41" "scikit-learn>=1.3" "pandas>=2.0" "tqdm>=4.66"

## C. Configuration  ← edit me

All paths, hyperparameters, and seeds. Nothing else needs editing.

In [ ]:
from pathlib import Path

# ----- Drive paths -----
UKP_CSV    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/UKP csv/8 UKP datasets.csv')
DIP_DIR    = Path('/content/drive/MyDrive/PhD Ali 26/Dataset/DIP2016')
OUTPUT_DIR = Path('/content/drive/MyDrive/PhD Ali 26/results/Ro2026')

# ----- Model & training -----
MODEL_NAME       = 'roberta-base'
MAX_LENGTH       = 128
EPOCHS           = 10
BATCH_SIZE       = 32
EVAL_BATCH_SIZE  = 64
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06
GRAD_CLIP        = 1.0
NUM_WORKERS      = 0

# ----- Experimental protocol -----
SEEDS         = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # paper uses 10 seeds
LABEL_SETUPS  = [2, 3]                            # both 2- and 3-label
MTL_MODES     = [True, False]                     # MTL+DIP2016 first, then single-task
TEST_TOPICS   = None                              # None = all 8 paper topics

# ----- MTL specifics -----
DIP_MAX_EXAMPLES = 300_000  # paper: 300K of 600K; set None to use all
QUERY_TEXT_CSV   = None     # optional [queryID,query_text] CSV; queryID used as text otherwise

# ----- Sanity check -----
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(exist_ok=True)
assert UKP_CSV.exists(), f'UKP CSV not found: {UKP_CSV}'
assert DIP_DIR.exists(), f'DIP folder not found: {DIP_DIR}'

print('UKP   :', UKP_CSV)
print('DIP   :', DIP_DIR)
print('OUT   :', OUTPUT_DIR)
print('Order :')
for use_mtl in MTL_MODES:
    for nl in LABEL_SETUPS:
        print(f'   {"MTL+DIP2016" if use_mtl else "single-task ":12s} / {nl}-label')

## D. Imports & globals

In [ ]:
import json, random, xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from functools import partial
from typing import Iterable

import numpy as np
import pandas as pd
import torch, torch.nn as nn
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    RobertaForSequenceClassification, RobertaModel, RobertaTokenizerFast,
    get_linear_schedule_with_warmup,
)
import matplotlib.pyplot as plt

TOPICS   = ['abortion', 'cloning', 'death penalty', 'gun control',
            'marijuana legalization', 'minimum wage', 'nuclear energy',
            'school uniforms']
LABELS_3 = ['NoArgument', 'Argument_against', 'Argument_for']
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

FIGS_DIR   = OUTPUT_DIR / 'figures'
TABLES_DIR = OUTPUT_DIR / 'tables'
print('device:', DEVICE)

## E. Data — UKP loader

Single CSV with columns `topic, sentence, annotation, set`. `annotation` ∈ {NoArgument,
Argument_for, Argument_against}; `set` ∈ {train, val, test}.

In [ ]:
@dataclass
class Example:
    topic: str
    sentence: str
    label: int

def label_to_id(annotation, num_labels):
    if num_labels == 3:
        return LABELS_3.index(annotation)
    return 0 if annotation == 'NoArgument' else 1

def load_ukp_csv(csv_path):
    df = pd.read_csv(csv_path)
    df['topic_norm'] = df['topic'].astype(str).str.strip().str.lower()
    return df

def build_ukp_splits(csv_path, test_topic, num_labels):
    df = load_ukp_csv(csv_path)
    t  = test_topic.strip().lower()
    train, val, test = [], [], []
    for _, row in df.iterrows():
        ex = Example(topic=str(row['topic']),
                     sentence=str(row['sentence']),
                     label=label_to_id(str(row['annotation']), num_labels))
        if row['topic_norm'] == t:
            if row['set'] == 'test': test.append(ex)
        else:
            if row['set'] == 'train': train.append(ex)
            elif row['set'] == 'val': val.append(ex)
    return {'train': train, 'val': val, 'test': test}

class UKPDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = list(examples); self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex.topic, ex.sentence, truncation=True,
                             max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

## F. Data — DIP2016 loader

Folder of XMLs of shape
`<singleQueryResults queryID="…"><documents><document><sentences>
<s relevant="true|false"><content>…`. Uses `queryID` as the topic text
when no `queryID → text` mapping CSV is provided.

In [ ]:
@dataclass
class DIPExample:
    query: str; sentence: str; label: int   # 1 = relevant

def _parse_dip_xml(path, query_text_map=None):
    root = ET.parse(path).getroot()
    qid  = root.attrib.get('queryID', path.stem)
    qtext = (query_text_map.get(str(qid)) if query_text_map else None) or str(qid)
    out = []
    for s in root.iter('s'):
        rel = s.attrib.get('relevant', 'false').strip().lower()
        c = s.find('content')
        if c is None or c.text is None: continue
        text = c.text.strip()
        if not text: continue
        out.append(DIPExample(query=qtext, sentence=text,
                              label=1 if rel == 'true' else 0))
    return out

def load_dip2016(dip_dir, query_text_map=None, max_examples=None):
    out = []
    for f in sorted(dip_dir.glob('*.xml')):
        out.extend(_parse_dip_xml(f, query_text_map))
        if max_examples is not None and len(out) >= max_examples:
            return out[:max_examples]
    return out

class DIPDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = list(examples); self.tokenizer = tokenizer; self.max_length = max_length
    def __len__(self):  return len(self.examples)
    def __getitem__(self, idx):
        ex = self.examples[idx]
        enc = self.tokenizer(ex.query, ex.sentence, truncation=True,
                             max_length=self.max_length, padding=False)
        item = {k: torch.as_tensor(v) for k, v in enc.items()}
        item['labels'] = torch.as_tensor(ex.label, dtype=torch.long)
        return item

# Optional queryID -> text mapping
QUERY_TEXT_MAP = {}
if QUERY_TEXT_CSV is not None and Path(QUERY_TEXT_CSV).exists():
    qdf = pd.read_csv(QUERY_TEXT_CSV)
    QUERY_TEXT_MAP = {str(q): str(t) for q, t in zip(qdf['queryID'], qdf['query_text'])}
    print(f'Loaded {len(QUERY_TEXT_MAP)} queryID->text mappings')
else:
    print('No queryID->text mapping; using queryID as topic text for DIP.')

## G. Collation

In [ ]:
def collate(batch, pad_token_id):
    L = max(x['input_ids'].size(0) for x in batch)
    out = {}
    for key in ('input_ids', 'attention_mask'):
        pad = pad_token_id if key == 'input_ids' else 0
        stacked = torch.full((len(batch), L), pad, dtype=torch.long)
        for i, x in enumerate(batch):
            stacked[i, :x[key].size(0)] = x[key]
        out[key] = stacked
    out['labels'] = torch.stack([x['labels'] for x in batch])
    return out

## H. Models

- **Single-task** = plain `RobertaForSequenceClassification`. Replaces the paper's `biclstm`.
- **MTL** = shared `RobertaModel` encoder + private UKP head + private DIP relevance head.
  Replaces the paper's `mtl+biclstm+dip2016`.

In [ ]:
def build_single_model(num_labels):
    tok = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
    mdl = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    return mdl, tok

class RobertaMTL(nn.Module):
    def __init__(self, main_num_labels, aux_num_labels=2, dropout=0.1):
        super().__init__()
        self.encoder   = RobertaModel.from_pretrained(MODEL_NAME)
        h              = self.encoder.config.hidden_size
        self.dropout   = nn.Dropout(dropout)
        self.main_head = nn.Linear(h, main_num_labels)
        self.aux_head  = nn.Linear(h, aux_num_labels)
        self.loss_fct  = nn.CrossEntropyLoss()
    def forward(self, input_ids, attention_mask, labels=None, task='main'):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])
        head   = self.main_head if task == 'main' else self.aux_head
        logits = head(pooled)
        loss   = self.loss_fct(logits, labels) if labels is not None else None
        return {'loss': loss, 'logits': logits}

def build_mtl_model(main_num_labels):
    tok = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
    return RobertaMTL(main_num_labels=main_num_labels), tok

## I. Metrics  (Table 4 columns)

- 2-label : `macro_f1`, `P_arg`, `R_arg`
- 3-label : `macro_f1`, `P_arg+`, `R_arg+`, `P_arg-`, `R_arg-`

In [ ]:
def compute_metrics(y_true, y_pred, num_labels):
    out = {'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0))}
    if num_labels == 2:
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], zero_division=0)
        out['P_arg'] = float(p[0]); out['R_arg'] = float(r[0])
    else:
        labels = [LABELS_3.index('Argument_for'), LABELS_3.index('Argument_against')]
        p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
        out['P_arg+'] = float(p[0]); out['R_arg+'] = float(r[0])
        out['P_arg-'] = float(p[1]); out['R_arg-'] = float(r[1])
    return out

NON_METRIC_KEYS = {'test_topic', 'seed', 'num_labels', 'use_mtl', 'model_name',
                   'train_losses', 'val_losses', 'y_true', 'y_pred'}

def aggregate(runs):
    keys = [k for k, v in runs[0].items()
            if k not in NON_METRIC_KEYS
            and isinstance(v, (int, float)) and not isinstance(v, bool)]
    return {k: (float(np.mean([r[k] for r in runs])),
                float(np.std([r[k] for r in runs]))) for k in keys}

## J. Training routines (with per-epoch checkpointing)

After every epoch, a single `ckpt.pt` (model + optim + sched + RNG + loss history)
is written to `OUTPUT_DIR/checkpoints/<tag>/`. On Colab disconnect just rerun
Section N — the in-flight run resumes at the next epoch. The ckpt is deleted
once the final per-run JSON lands on Drive.

In [ ]:
@dataclass
class HParams:
    test_topic: str = 'gun control'
    num_labels: int = 2
    seed:       int = 0
    use_mtl:    bool = False
    epochs:     int  = EPOCHS

def run_tag(hp):
    mtl = 'mtl' if hp.use_mtl else 'single'
    return f"{mtl}_L{hp.num_labels}_{hp.test_topic.replace(' ', '_')}_seed{hp.seed}"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def _make_optim(model, total_steps):
    nd = ('bias', 'LayerNorm.weight')
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(x in n for x in nd)],
         'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if     any(x in n for x in nd)],
         'weight_decay': 0.0},
    ]
    optim = AdamW(groups, lr=LR)
    sched = get_linear_schedule_with_warmup(optim, int(total_steps * WARMUP_RATIO), total_steps)
    return optim, sched

def _ckpt_path(hp):
    return OUTPUT_DIR / 'checkpoints' / run_tag(hp) / 'ckpt.pt'

def _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state, train_losses, val_losses):
    p = _ckpt_path(hp); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix('.pt.tmp')
    torch.save({
        'model': model.state_dict(), 'optim': optim.state_dict(), 'sched': sched.state_dict(),
        'epoch': epoch, 'best_val_loss': best_val, 'best_state': best_state,
        'train_losses': train_losses, 'val_losses': val_losses,
        'rng_torch': torch.get_rng_state(),
        'rng_cuda':  torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_numpy': np.random.get_state(), 'rng_python': random.getstate(),
    }, tmp)
    tmp.replace(p)

def _load_ckpt(hp, model, optim, sched):
    p = _ckpt_path(hp)
    if not p.exists():
        return 0, float('inf'), None, [], []
    data = torch.load(p, map_location=DEVICE, weights_only=False)
    model.load_state_dict(data['model'])
    optim.load_state_dict(data['optim'])
    sched.load_state_dict(data['sched'])
    torch.set_rng_state(data['rng_torch'].cpu())
    if data.get('rng_cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in data['rng_cuda']])
    np.random.set_state(data['rng_numpy'])
    random.setstate(data['rng_python'])
    print(f"  resuming {run_tag(hp)}: epoch {data['epoch']+1}/{hp.epochs} "
          f"(best val loss {data['best_val_loss']:.4f})")
    return (data['epoch']+1, data['best_val_loss'], data['best_state'],
            data.get('train_losses', []), data.get('val_losses', []))

def _cleanup_ckpt(hp):
    p = _ckpt_path(hp)
    if p.exists(): p.unlink()
    try: p.parent.rmdir()
    except OSError: pass

@torch.no_grad()
def _evaluate(model, loader, mtl_task=None):
    model.eval()
    losses, preds, golds = [], [], []
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        if mtl_task is None:
            out = model(**batch); loss, logits = out.loss, out.logits
        else:
            out = model(**batch, task=mtl_task); loss, logits = out['loss'], out['logits']
        losses.append(loss.item() * batch['labels'].size(0))
        preds.append(logits.argmax(-1).cpu().numpy())
        golds.append(batch['labels'].cpu().numpy())
    n = sum(len(g) for g in golds)
    return sum(losses) / max(n, 1), np.concatenate(preds), np.concatenate(golds)

def _train_one(model, loader, optim, sched, task=None, pbar_desc=''):
    model.train()
    pbar = tqdm(loader, desc=pbar_desc, position=2, leave=False)
    s, n = 0.0, 0
    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch, task=task) if task is not None else model(**batch)
        loss = out['loss'] if task is not None else out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optim.step(); sched.step(); optim.zero_grad()
        bsz = batch['labels'].size(0); s += float(loss.item()) * bsz; n += bsz
        pbar.set_postfix(loss=float(loss.item()))
    return s / max(n, 1)

def run_one(hp):
    if not _ckpt_path(hp).exists():
        set_seed(hp.seed)

    splits = build_ukp_splits(UKP_CSV, hp.test_topic, hp.num_labels)
    if hp.use_mtl:
        model, tok = build_mtl_model(hp.num_labels)
    else:
        model, tok = build_single_model(hp.num_labels)
    model.to(DEVICE)

    coll = partial(collate, pad_token_id=tok.pad_token_id)
    train_loader = DataLoader(UKPDataset(splits['train'], tok, MAX_LENGTH),
                              batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=coll, num_workers=NUM_WORKERS)
    val_loader   = DataLoader(UKPDataset(splits['val'],   tok, MAX_LENGTH),
                              batch_size=EVAL_BATCH_SIZE, collate_fn=coll)
    test_loader  = DataLoader(UKPDataset(splits['test'],  tok, MAX_LENGTH),
                              batch_size=EVAL_BATCH_SIZE, collate_fn=coll)

    if hp.use_mtl:
        dip_examples = load_dip2016(DIP_DIR, QUERY_TEXT_MAP or None,
                                    max_examples=DIP_MAX_EXAMPLES)
        aux_loader = DataLoader(DIPDataset(dip_examples, tok, MAX_LENGTH),
                                batch_size=BATCH_SIZE, shuffle=True,
                                collate_fn=coll, num_workers=NUM_WORKERS)
        total_steps = (len(train_loader) + len(aux_loader)) * hp.epochs
    else:
        aux_loader  = None
        total_steps = len(train_loader) * hp.epochs

    optim, sched = _make_optim(model, total_steps)
    start_epoch, best_val, best_state, train_losses, val_losses = \
        _load_ckpt(hp, model, optim, sched)

    ebar = tqdm(total=hp.epochs, initial=start_epoch,
                desc=f'{run_tag(hp)}', unit='ep', position=1, leave=True)
    for epoch in range(start_epoch, hp.epochs):
        if hp.use_mtl:
            _train_one(model, aux_loader, optim, sched, task='aux',
                       pbar_desc=f'ep {epoch+1}/{hp.epochs} aux')
            tr_loss = _train_one(model, train_loader, optim, sched, task='main',
                                 pbar_desc=f'ep {epoch+1}/{hp.epochs} main')
            val_loss, _, _ = _evaluate(model, val_loader, mtl_task='main')
        else:
            tr_loss = _train_one(model, train_loader, optim, sched,
                                 pbar_desc=f'ep {epoch+1}/{hp.epochs}')
            val_loss, _, _ = _evaluate(model, val_loader)
        train_losses.append(tr_loss); val_losses.append(val_loss)
        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        _save_ckpt(hp, model, optim, sched, epoch, best_val, best_state,
                   train_losses, val_losses)
        ebar.set_postfix(train=f'{tr_loss:.4f}', val=f'{val_loss:.4f}', best=f'{best_val:.4f}')
        ebar.update(1)
    ebar.close()

    if best_state is not None:
        model.load_state_dict(best_state)
    _, preds, golds = _evaluate(model, test_loader,
                                mtl_task='main' if hp.use_mtl else None)
    metrics = compute_metrics(golds, preds, hp.num_labels)
    metrics.update({
        'best_val_loss': best_val,
        'train_losses': train_losses, 'val_losses': val_losses,
        'y_true': golds.tolist(), 'y_pred': preds.tolist(),
        'test_topic': hp.test_topic, 'seed': hp.seed,
        'num_labels': hp.num_labels, 'use_mtl': hp.use_mtl,
        'model_name': MODEL_NAME,
    })
    _cleanup_ckpt(hp)
    return metrics

## K. Sanity check (small smoke test)

In [ ]:
print('UKP rows:', len(load_ukp_csv(UKP_CSV)))
dip_sample = load_dip2016(DIP_DIR, QUERY_TEXT_MAP or None, max_examples=200)
print(f'DIP sample: {len(dip_sample)} sentences (first one): '
      f'query={dip_sample[0].query!r}, label={dip_sample[0].label}, '
      f'sent={dip_sample[0].sentence[:70]!r}')

## L. Optional smoke run (≈ 3 min on T4)

1 topic, 1 seed, 2 epochs, single-task. Verifies the pipeline before launching the grid.

In [ ]:
smoke_hp = HParams(test_topic='gun control', num_labels=2, seed=0, use_mtl=False, epochs=2)
smoke = run_one(smoke_hp)
print({k: v for k, v in smoke.items()
       if k in {'macro_f1', 'P_arg', 'R_arg', 'best_val_loss'}})

## M. Full grid — MTL+DIP2016 first, then single-task

For every `(use_mtl, num_labels, topic, seed)`:

- if its JSON already exists on Drive → skip (already done)
- else → train, test, write `<tag>.json`

A green ✓ is printed the moment the last seed of a held-out topic finishes,
so you can watch dataset-level progress in real time.

In [ ]:
topics_to_run = TEST_TOPICS or TOPICS

plan = [(use_mtl, num_labels, topic, seed)
        for use_mtl    in MTL_MODES
        for num_labels in LABEL_SETUPS
        for topic      in topics_to_run
        for seed       in SEEDS]
def _tag(use_mtl, num_labels, topic, seed):
    return f"{'mtl' if use_mtl else 'single'}_L{num_labels}_{topic.replace(' ', '_')}_seed{seed}"

done_at_start = sum(1 for c in plan
                    if (OUTPUT_DIR / f'{_tag(*c)}.json').exists())
print(f'Grid: {len(plan)} runs total, {done_at_start} done on Drive, '
      f'{len(plan) - done_at_start} remaining.')

grid_bar = tqdm(total=len(plan), initial=done_at_start,
                desc='grid', unit='run', position=0, leave=True)
topic_seeds: dict[tuple, set] = {}
ticked: set = set()
results: dict[tuple, list[dict]] = {}

for combo in plan:
    use_mtl, num_labels, topic, seed = combo
    tag = _tag(*combo)
    out_path = OUTPUT_DIR / f'{tag}.json'
    grid_bar.set_postfix_str(tag)
    if out_path.exists():
        res = json.loads(out_path.read_text())
    else:
        res = run_one(HParams(test_topic=topic, num_labels=num_labels,
                              seed=seed, use_mtl=use_mtl))
        out_path.write_text(json.dumps(res, indent=2))
        grid_bar.update(1)
    results.setdefault((use_mtl, num_labels, topic), []).append(res)

    key = (use_mtl, num_labels, topic)
    topic_seeds.setdefault(key, set()).add(seed)
    if len(topic_seeds[key]) == len(SEEDS) and key not in ticked:
        ticked.add(key)
        tqdm.write(f'  ✓  {"mtl+dip" if use_mtl else "single  "} / '
                   f'{num_labels}-label / {topic}  '
                   f'({len(SEEDS)}/{len(SEEDS)} seeds done)')
grid_bar.close()

# Aggregate and write summary.json
summary = {}
for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        per_topic = {t: results[(use_mtl, num_labels, t)]
                     for t in topics_to_run if (use_mtl, num_labels, t) in results}
        if not per_topic: continue
        key = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        summary[key] = {t: aggregate(r) for t, r in per_topic.items()}
        all_runs   = [r for rs in per_topic.values() for r in rs]
        summary[key]['__overall__'] = aggregate(all_runs)
(OUTPUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))

# Final dataset checklist
print('\n=== Final dataset checklist ===')
for use_mtl in MTL_MODES:
    print(f'\n[{"MTL + DIP2016" if use_mtl else "single-task (no DIP)"}]')
    for num_labels in LABEL_SETUPS:
        print(f'  {num_labels}-label:')
        for topic in topics_to_run:
            done = len(topic_seeds.get((use_mtl, num_labels, topic), set()))
            mark = '✓' if done == len(SEEDS) else f'{done}/{len(SEEDS)}'
            print(f'    {mark}  {topic}')

## N. Load results (used by all analysis sections below)

Re-runnable any time; gathers every per-run JSON written so far.

In [ ]:
def load_all_results():
    out = []
    for p in sorted(OUTPUT_DIR.glob('*_seed*.json')):
        try: out.append(json.loads(p.read_text()))
        except Exception: pass
    return out

ALL_RUNS = load_all_results()
PRED_RUNS = [r for r in ALL_RUNS if 'y_true' in r and 'y_pred' in r]
print(f'Loaded {len(ALL_RUNS)} run JSONs, {len(PRED_RUNS)} with predictions.')

## O. Summary tables (CSV)

- `tables/summary.csv` — aggregated mean ± std per (setup, scope, metric)
- `tables/all_runs.csv` — one row per completed run

In [ ]:
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())

for setup, by_topic in summary.items():
    print(f'\n=== {setup} ===')
    for k, (m, s) in by_topic['__overall__'].items():
        print(f'  {k:<14s} {m:.4f} ± {s:.4f}')

rows = []
for setup, by_topic in summary.items():
    for scope, metrics in by_topic.items():
        for metric, (mean, std) in metrics.items():
            rows.append({'setup': setup, 'scope': scope,
                         'metric': metric, 'mean': mean, 'std': std})
pd.DataFrame(rows).to_csv(TABLES_DIR / 'summary.csv', index=False)

# Per-run table
mkeys = ['macro_f1', 'P_arg', 'R_arg', 'P_arg+', 'P_arg-', 'R_arg+', 'R_arg-', 'best_val_loss']
rrows = []
for r in ALL_RUNS:
    rrows.append({
        'mode': 'mtl' if r.get('use_mtl') else 'single',
        'num_labels': r.get('num_labels'), 'test_topic': r.get('test_topic'),
        'seed': r.get('seed'),
        **{k: r.get(k) for k in mkeys},
        'n_epochs': len(r.get('train_losses') or []),
    })
if rrows:
    pd.DataFrame(rrows).sort_values(['mode', 'num_labels', 'test_topic', 'seed']) \
        .to_csv(TABLES_DIR / 'all_runs.csv', index=False)

print('wrote', TABLES_DIR / 'summary.csv')
print('wrote', TABLES_DIR / 'all_runs.csv')

## P. Paper-style metric tables (Stab et al. Table 4)

Rows = topics + OVERALL. Columns:
- 2-label : `F1`, `P_arg`, `R_arg`
- 3-label : `F1`, `P_arg+`, `P_arg-`, `R_arg+`, `R_arg-`

In [ ]:
PAPER_COLS = {
    2: [('F1', 'macro_f1'), ('P_arg', 'P_arg'), ('R_arg', 'R_arg')],
    3: [('F1', 'macro_f1'),
        ('P_arg+', 'P_arg+'), ('P_arg-', 'P_arg-'),
        ('R_arg+', 'R_arg+'), ('R_arg-', 'R_arg-')],
}
topics = TEST_TOPICS or TOPICS

combined = []
for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        runs  = [r for r in ALL_RUNS
                 if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        if not runs: continue
        cols  = PAPER_COLS[num_labels]
        fmt_rows, num_rows = [], []
        for topic in topics + ['OVERALL']:
            scope_runs = runs if topic == 'OVERALL' else [r for r in runs if r.get('test_topic') == topic]
            if not scope_runs: continue
            frow = {'topic': topic}
            nrow = {'topic': topic, 'n_runs': len(scope_runs)}
            for name, key in cols:
                vals = [r[key] for r in scope_runs if key in r]
                if vals:
                    m, s = float(np.mean(vals)), float(np.std(vals))
                    frow[name] = f'{m:.4f} ± {s:.4f}'
                    nrow[f'{name}_mean'] = m; nrow[f'{name}_std'] = s
                else:
                    frow[name] = '—'
            fmt_rows.append(frow); num_rows.append(nrow)
            combined.append({'setup': setup, **nrow})
        fmt_df = pd.DataFrame(fmt_rows, columns=['topic', *[c for c, _ in cols]])
        num_df = pd.DataFrame(num_rows)
        print(f'\n=== {setup} — Table 4 style ===')
        print(fmt_df.to_string(index=False))
        fmt_df.to_csv(TABLES_DIR / f'paper_table_{setup}.csv', index=False)
        num_df.to_csv(TABLES_DIR / f'paper_table_{setup}_numeric.csv', index=False)

pd.DataFrame(combined).to_csv(TABLES_DIR / 'paper_table_all.csv', index=False)
print('\nwrote', TABLES_DIR / 'paper_table_all.csv')

## Q. Confusion matrices (PNG + CSV)

Per held-out topic and OVERALL, for every `(mode, num_labels)`. Predictions are
pooled across seeds.

In [ ]:
def label_names(n): return ['NoArg', 'Arg'] if n == 2 else ['NoArg', 'Arg-', 'Arg+']

def plot_cm(ax, cm, names, title):
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title, fontsize=10)
    thr = cm.max() / 2 if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > thr else 'black', fontsize=8)

topics = TEST_TOPICS or TOPICS
for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        names = label_names(num_labels)
        runs  = [r for r in PRED_RUNS
                 if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        if not runs:
            print(f'(skip {setup})'); continue

        ncols = 4; nrows = (len(topics) + ncols) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
        axes = axes.flatten()
        all_yt, all_yp = [], []
        for ax, topic in zip(axes, topics):
            yt, yp = [], []
            for r in runs:
                if r.get('test_topic') == topic:
                    yt += r['y_true']; yp += r['y_pred']
            all_yt += yt; all_yp += yp
            if yt:
                cm = confusion_matrix(yt, yp, labels=list(range(num_labels)))
                plot_cm(ax, cm, names, f'{topic}\n(n={len(yt)})')
                pd.DataFrame(cm, index=[f'true_{n}' for n in names],
                             columns=[f'pred_{n}' for n in names]) \
                  .to_csv(TABLES_DIR / f"cm_{setup}_{topic.replace(' ', '_')}.csv")
            else:
                ax.set_visible(False)

        ax = axes[len(topics)] if len(topics) < len(axes) else axes[-1]
        if all_yt:
            cm = confusion_matrix(all_yt, all_yp, labels=list(range(num_labels)))
            plot_cm(ax, cm, names, f'OVERALL\n(n={len(all_yt)})')
            pd.DataFrame(cm, index=[f'true_{n}' for n in names],
                         columns=[f'pred_{n}' for n in names]) \
              .to_csv(TABLES_DIR / f'cm_{setup}_OVERALL.csv')
        for ax in axes[len(topics) + 1:]:
            ax.set_visible(False)
        fig.suptitle(f"{'MTL+DIP' if use_mtl else 'single-task'} — {num_labels}-label", fontsize=14)
        fig.tight_layout()
        fig.savefig(FIGS_DIR / f'cm_{setup}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('wrote', FIGS_DIR / f'cm_{setup}.png')

## R. Train / val loss curves (overfitting view)

Averaged over runs of each setup, ± 1 std band.

In [ ]:
def stack_curves(curves):
    if not curves: return np.array([]), np.array([])
    L = max(len(c) for c in curves)
    arr = np.full((len(curves), L), np.nan)
    for i, c in enumerate(curves): arr[i, :len(c)] = c
    return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)

setups = [(m, n) for m in MTL_MODES for n in LABEL_SETUPS]
fig, axes = plt.subplots(1, len(setups), figsize=(5*len(setups), 4), squeeze=False)
axes = axes.flatten()
crows = []
for ax, (use_mtl, num_labels) in zip(axes, setups):
    setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
    runs  = [r for r in ALL_RUNS
             if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels
             and 'train_losses' in r and 'val_losses' in r]
    if not runs:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                transform=ax.transAxes); ax.set_axis_off(); continue
    trm, trs = stack_curves([r['train_losses'] for r in runs])
    vam, vas = stack_curves([r['val_losses']   for r in runs])
    x = np.arange(1, len(trm)+1)
    ax.plot(x, trm, label='train', color='tab:blue')
    ax.fill_between(x, trm-trs, trm+trs, alpha=0.2, color='tab:blue')
    ax.plot(x, vam, label='val',   color='tab:orange')
    ax.fill_between(x, vam-vas, vam+vas, alpha=0.2, color='tab:orange')
    ax.set_xlabel('epoch'); ax.set_ylabel('CE loss'); ax.grid(alpha=0.3)
    ax.set_title(f'{"MTL+DIP" if use_mtl else "single"} / {num_labels}-label\n(avg of {len(runs)} runs)')
    ax.legend()
    for i, ep in enumerate(x):
        crows.append({'setup': setup, 'epoch': int(ep),
                      'train_mean': float(trm[i]), 'train_std': float(trs[i]),
                      'val_mean':   float(vam[i]), 'val_std':   float(vas[i])})
fig.suptitle('Average train/val loss (± 1 std)', fontsize=14)
fig.tight_layout()
fig.savefig(FIGS_DIR / 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
if crows:
    pd.DataFrame(crows).to_csv(TABLES_DIR / 'loss_curves.csv', index=False)
    print('wrote', TABLES_DIR / 'loss_curves.csv')

## S. Classification reports  (CSV + TXT)

`sklearn.metrics.classification_report` per topic and overall, per setup.

In [ ]:
text_lines, combined = [], []
def report_rows(setup, scope, yt, yp, num_labels, names):
    d = classification_report(yt, yp, labels=list(range(num_labels)),
                              target_names=names, output_dict=True, zero_division=0)
    rows = []
    for cls, vals in d.items():
        if isinstance(vals, dict):
            rows.append({'setup': setup, 'scope': scope, 'class': cls,
                         'precision': vals.get('precision'),
                         'recall':    vals.get('recall'),
                         'f1':        vals.get('f1-score'),
                         'support':   vals.get('support')})
        else:
            rows.append({'setup': setup, 'scope': scope, 'class': cls,
                         'precision': None, 'recall': None,
                         'f1': float(vals), 'support': None})
    return rows

topics = TEST_TOPICS or TOPICS
for use_mtl in MTL_MODES:
    for num_labels in LABEL_SETUPS:
        names = label_names(num_labels)
        runs  = [r for r in PRED_RUNS
                 if r.get('use_mtl') == use_mtl and r.get('num_labels') == num_labels]
        if not runs: continue
        setup = f"{'mtl' if use_mtl else 'single'}_{num_labels}label"
        header = f"\n{'='*72}\n{setup.upper()}\n{'='*72}"
        text_lines.append(header); print(header)
        all_yt, all_yp = [], []
        for topic in topics:
            yt, yp = [], []
            for r in runs:
                if r.get('test_topic') == topic:
                    yt += r['y_true']; yp += r['y_pred']
            all_yt += yt; all_yp += yp
            if not yt: continue
            sub = f"\n--- {topic}  (n={len(yt)}) ---"
            rep = classification_report(yt, yp, labels=list(range(num_labels)),
                                        target_names=names, digits=4, zero_division=0)
            text_lines += [sub, rep]; print(sub); print(rep)
            rows = report_rows(setup, topic, yt, yp, num_labels, names)
            combined += rows
            pd.DataFrame(rows).to_csv(
                TABLES_DIR / f"cls_{setup}_{topic.replace(' ', '_')}.csv", index=False)
        if all_yt:
            sub = f"\n--- OVERALL  (n={len(all_yt)}) ---"
            rep = classification_report(all_yt, all_yp, labels=list(range(num_labels)),
                                        target_names=names, digits=4, zero_division=0)
            text_lines += [sub, rep]; print(sub); print(rep)
            rows = report_rows(setup, 'OVERALL', all_yt, all_yp, num_labels, names)
            combined += rows
            pd.DataFrame(rows).to_csv(TABLES_DIR / f'cls_{setup}_OVERALL.csv', index=False)

(OUTPUT_DIR / 'classification_reports.txt').write_text('\n'.join(text_lines))
pd.DataFrame(combined).to_csv(TABLES_DIR / 'classification_reports.csv', index=False)
print('\nwrote', TABLES_DIR / 'classification_reports.csv')